In [1]:
# CELL 1 — Verify GPU
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ CUDA not available — check your install before continuing")

PyTorch version : 2.0.1+cu117
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2050
VRAM            : 4.3 GB


In [2]:
# CELL 2 — All paths and config in one place
import os

# ── Adjust these two lines to match where you saved your files ──────────────
MS2_DATA_DIR  = r"D:\Undergrad Research\bart\ms2_backup\ms2_data"   # folder with the 3 jsonl files
MODEL_SAVE_DIR = r"D:\Undergrad Research\bart\output"     # where checkpoints & final model go
# ─────────────────────────────────────────────────────────────────────────────

MS2_TRAIN      = os.path.join(MS2_DATA_DIR, "training_reviews.jsonl")
MS2_VAL        = os.path.join(MS2_DATA_DIR, "validation_reviews.jsonl")
MS2_TEST       = os.path.join(MS2_DATA_DIR, "testing_reviews.jsonl")
CHECKPOINT_DIR = os.path.join(MODEL_SAVE_DIR, "checkpoints")
FINAL_MODEL    = os.path.join(MODEL_SAVE_DIR, "final_model")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL, exist_ok=True)

checks = {
    "MS2 train" : MS2_TRAIN,
    "MS2 val"   : MS2_VAL,
    "MS2 test"  : MS2_TEST,
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌ MISSING'}  {name}: {path}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("✅ All paths verified — ready to proceed")
else:
    print("❌ Fix missing paths before continuing")
    print("   Make sure you copied the 3 jsonl files from Drive to MS2_DATA_DIR")

  ✅  MS2 train: D:\Undergrad Research\bart\ms2_backup\ms2_data\training_reviews.jsonl
  ✅  MS2 val: D:\Undergrad Research\bart\ms2_backup\ms2_data\validation_reviews.jsonl
  ✅  MS2 test: D:\Undergrad Research\bart\ms2_backup\ms2_data\testing_reviews.jsonl

✅ All paths verified — ready to proceed


In [3]:
# CELL 3 — Load MS2 and build proper source/target pairs
import json, re, random
import pandas as pd

random.seed(42)

def load_jsonl_sample(path, max_samples):
    records = []
    filename = os.path.basename(path)
    print(f"  Reading {filename}...")
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
            if len(records) >= max_samples:
                break
    print(f"  → {len(records)} records loaded")
    return records

def clean(text):
    text = re.sub(r'<[^>]+>', '', str(text))
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_source(row):
    """
    Multi-document source: title + abstract for each included study.
    This is the FIXED version — not just titles.
    """
    parts = []
    for study in row.get('included_studies', []):
        for ref in study.get('references', []):
            title    = clean(ref.get('title', ''))
            abstract = ''
            meta = ref.get('metadata', {})
            if isinstance(meta, dict):
                for k, v in meta.items():
                    if isinstance(v, str) and len(v) > 40:
                        abstract = clean(v)
                        break
            if title and abstract:
                parts.append(f"{title}. {abstract}")
            elif title:
                parts.append(title)
            elif abstract:
                parts.append(abstract)

    # fallback if abstracts are sparse
    if len(' '.join(parts).split()) < 80:
        for field in ['interventions', 'outcomes', 'populations']:
            items = row.get(field, [])
            if isinstance(items, list):
                parts.extend([clean(x) for x in items if len(clean(x)) > 15])

    return ' '.join(parts).strip()

def build_target(row):
    """Target = the Cochrane review abstract."""
    abstract = row.get('abstract', '')
    if isinstance(abstract, str) and len(abstract) > 50:
        return clean(abstract)
    structured = row.get('structured_abstract', [])
    if isinstance(structured, list):
        parts = []
        for item in structured:
            if isinstance(item, list) and len(item) >= 2:
                parts.append(clean(str(item[1])))
        joined = ' '.join(parts)
        if len(joined) > 50:
            return joined
    return ''

def flatten(records, split_name):
    rows, skipped = [], 0
    for row in records:
        src = build_source(row)
        tgt = build_target(row)
        src_words = len(src.split())
        tgt_words = len(tgt.split())
        if src_words >= 150 and tgt_words >= 50 and src_words > tgt_words:
            rows.append({'source': src, 'target': tgt})
        else:
            skipped += 1
    df = pd.DataFrame(rows)
    if len(df) > 0:
        ratio    = (df['target'].apply(lambda x: len(x.split())) /
                    df['source'].apply(lambda x: len(x.split()))).mean()
        src_mean = df['source'].apply(lambda x: len(x.split())).mean()
        tgt_mean = df['target'].apply(lambda x: len(x.split())).mean()
        print(f"  {split_name}: kept {len(df)}, skipped {skipped}")
        print(f"    avg src: {src_mean:.0f} words | avg tgt: {tgt_mean:.0f} words | "
              f"compression ratio: {ratio:.3f}  ← must be < 1.0")
    else:
        print(f"  {split_name}: ❌ 0 rows kept — something is wrong with extraction")
    return df

print("Loading MS2 records...")
train_raw = load_jsonl_sample(MS2_TRAIN, max_samples=10000)
val_raw   = load_jsonl_sample(MS2_VAL,   max_samples=1000)
test_raw  = load_jsonl_sample(MS2_TEST,  max_samples=1000)

print("\nFlattening into source/target pairs...")
ms2_train = flatten(train_raw, 'TRAIN')
ms2_val   = flatten(val_raw,   'VAL')
ms2_test  = flatten(test_raw,  'TEST')

# Sanity check — read these carefully
print("\n--- SOURCE SAMPLE (first 600 chars) ---")
print(ms2_train['source'].iloc[0][:600])
print("\n--- TARGET SAMPLE (first 400 chars) ---")
print(ms2_train['target'].iloc[0][:400])

# Save to disk immediately
ms2_train.to_csv(os.path.join(MODEL_SAVE_DIR, 'ms2_train_processed.csv'), index=False)
ms2_val.to_csv(os.path.join(MODEL_SAVE_DIR,   'ms2_val_processed.csv'),   index=False)
ms2_test.to_csv(os.path.join(MODEL_SAVE_DIR,  'ms2_test_processed.csv'),  index=False)
print("\n✅ Processed CSVs saved")

Loading MS2 records...
  Reading training_reviews.jsonl...
  → 10000 records loaded
  Reading validation_reviews.jsonl...
  → 1000 records loaded
  Reading testing_reviews.jsonl...
  → 1000 records loaded

Flattening into source/target pairs...
  TRAIN: kept 4269, skipped 5731
    avg src: 666 words | avg tgt: 336 words | compression ratio: 0.613  ← must be < 1.0
  VAL: kept 478, skipped 522
    avg src: 632 words | avg tgt: 322 words | compression ratio: 0.606  ← must be < 1.0
  TEST: kept 487, skipped 513
    avg src: 668 words | avg tgt: 353 words | compression ratio: 0.630  ← must be < 1.0

--- SOURCE SAMPLE (first 600 chars) ---
Large volumes of apple juice preoperatively do not affect gastric pH and volume in children. Canadian journal of anaesthesia = Journal canadien d'anesthesie The effect of preoperative carbohydrate loading on hormonal changes, hepatic glycogen, and glucoregulatory enzymes during abdominal surgery. The effect of preoperative apple juice on gastric contents, 

In [4]:
# CELL 4 — Load BART and tokenize
from transformers import BartTokenizer, BartForConditionalGeneration
from datasets import Dataset
import torch

BASE_MODEL = "facebook/bart-base"
print(f"Loading {BASE_MODEL}...")
tokenizer = BartTokenizer.from_pretrained(BASE_MODEL)
model     = BartForConditionalGeneration.from_pretrained(BASE_MODEL)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print(f"✅ Model loaded")
print(f"   Parameters : {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"   Device     : {device}")

# 4GB VRAM — keep input at 512
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 192

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch['source'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding='max_length'
    )
    labels = tokenizer(
        text_target=batch['target'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding='max_length'
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

print("\nTokenizing...")
train_ds  = Dataset.from_pandas(ms2_train[['source','target']].reset_index(drop=True))
val_ds    = Dataset.from_pandas(ms2_val[['source','target']].reset_index(drop=True))
test_ds   = Dataset.from_pandas(ms2_test[['source','target']].reset_index(drop=True))

train_tok = train_ds.map(tokenize_batch, batched=True, batch_size=8,
                         remove_columns=['source','target'])
val_tok   = val_ds.map(tokenize_batch,   batched=True, batch_size=8,
                       remove_columns=['source','target'])
test_tok  = test_ds.map(tokenize_batch,  batched=True, batch_size=8,
                        remove_columns=['source','target'])

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

# Save tokenized datasets to disk
train_tok.save_to_disk(os.path.join(MODEL_SAVE_DIR, 'tok_train'))
val_tok.save_to_disk(os.path.join(MODEL_SAVE_DIR,   'tok_val'))
test_tok.save_to_disk(os.path.join(MODEL_SAVE_DIR,  'tok_test'))

print(f"\n✅ Tokenization done")
print(f"   Train : {len(train_tok)}")
print(f"   Val   : {len(val_tok)}")
print(f"   Test  : {len(test_tok)}")
print("✅ Tokenized datasets saved to disk")

Loading facebook/bart-base...


D:\Undergrad Research\bart\venv311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Model loaded
   Parameters : 139.4M
   Device     : cuda

Tokenizing...


Map:   0%|          | 0/4269 [00:00<?, ? examples/s]

Map:   0%|          | 0/478 [00:00<?, ? examples/s]

Map:   0%|          | 0/487 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4269 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/478 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/487 [00:00<?, ? examples/s]


✅ Tokenization done
   Train : 4269
   Val   : 478
   Test  : 487
✅ Tokenized datasets saved to disk


In [ ]:
# CELL 5 — Fine-tune BART on MS2
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
import evaluate, numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    vocab_size = tokenizer.vocab_size
    preds  = np.clip(preds, 0, vocab_size - 1).astype(np.int32)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, vocab_size - 1).astype(np.int32)
    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,
                                                skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels,
                                                skip_special_tokens=True)]
    result = rouge.compute(predictions=decoded_preds,
                           references=decoded_labels,
                           use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    num_train_epochs            = 6,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 8,
    warmup_steps                = 200,
    weight_decay                = 0.01,
    learning_rate               = 3e-5,
    lr_scheduler_type           = "cosine",
    fp16                        = True,
    predict_with_generate       = True,
    generation_max_length       = 192,
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "rougeL",
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = "none",
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = val_tok,
    tokenizer       = tokenizer,       # ← fixed
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print("🚀 Starting training...")
print(f"   Train samples  : {len(train_tok)}")
print(f"   Val samples    : {len(val_tok)}")
print(f"   Effective batch: 16")
print(f"   Max epochs     : 6")
print(f"   Device         : {device}")
print(f"   Checkpoints → {CHECKPOINT_DIR}\n")

trainer.train()
print("✅ Training complete!")

🚀 Starting training...
   Train samples  : 4269
   Val samples    : 478
   Effective batch: 16
   Max epochs     : 6
   Device         : cuda
   Checkpoints → D:\Undergrad Research\bart\output\checkpoints



Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
0,2.821800,2.755763,0.348000,0.085100,0.191800,0.192000
1,2.837000,2.661079,0.352200,0.089700,0.190800,0.190900


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


In [6]:
# CELL 6 — Save best model
model.save_pretrained(FINAL_MODEL)
tokenizer.save_pretrained(FINAL_MODEL)

import os
print(f"✅ Final model saved to: {FINAL_MODEL}")
print(f"   Files: {os.listdir(FINAL_MODEL)}")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


✅ Final model saved to: D:\Undergrad Research\bart\output\final_model
   Files: ['config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer_config.json', 'vocab.json']


In [7]:
# CELL 7 — Test set evaluation
import numpy as np, evaluate, json

rouge_eval = evaluate.load("rouge")

print("Running evaluation on test set...")
test_results = trainer.predict(test_tok)

preds  = test_results.predictions
labels = test_results.label_ids

vocab_size = tokenizer.vocab_size
preds  = np.clip(preds, 0, vocab_size - 1).astype(np.int32)
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
labels = np.clip(labels, 0, vocab_size - 1).astype(np.int32)

decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,
                                            skip_special_tokens=True)]
decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels,
                                            skip_special_tokens=True)]

scores = rouge_eval.compute(predictions=decoded_preds,
                            references=decoded_labels,
                            use_stemmer=True)

print("\n" + "="*45)
print("      FINAL TEST SET ROUGE SCORES")
print("="*45)
for k, v in scores.items():
    print(f"  {k:12s}: {v:.4f}")
print("="*45)

# Show 3 examples
print("\n--- QUALITATIVE EXAMPLES ---")
for i in range(3):
    print(f"\n[Example {i+1}]")
    print(f"  PRED : {decoded_preds[i][:300]}")
    print(f"  REF  : {decoded_labels[i][:300]}")

# Save scores
scores_path = os.path.join(MODEL_SAVE_DIR, 'test_rouge_scores.json')
with open(scores_path, 'w') as f:
    json.dump(scores, f, indent=2)
print(f"\n✅ Scores saved to {scores_path}")

Running evaluation on test set...



      FINAL TEST SET ROUGE SCORES
  rouge1      : 0.3666
  rouge2      : 0.0974
  rougeL      : 0.2000
  rougeLsum   : 0.2000

--- QUALITATIVE EXAMPLES ---

[Example 1]
  PRED : BACKGROUND Postoperative adjuvant chemotherapy ( PVM ) has been shown to improve the survival of patients with stage I non-small cell lung cancer ( NSCLC ). OBJECTIVES To assess the effect of PVM on progression-free survival ( PFS ) in patients with non-NSCLC. SEARCH METHODS We search ed the Cochran
  REF  : Adjuvant chemotherapy is associated with increased overall survival in non-small cell lung cancer ( NSCLC ), but is associated with high- grade toxicity. The effect of cisplatin-based adjuvant chemotherapy on non-lung cancer-related mortality is not well investigated. We conducted a systematic revie

[Example 2]
  PRED : BACKGROUND Stroke is the most common cause of disability in the United States. The aim of this systematic review was to assess the effect of exercise on the quality of life after stroke. M

In [8]:
# CELL 8 — Test inference on a custom input
import torch

model.eval()
model.to(device)

test_text = """
A randomized controlled trial assessed the efficacy of glucocorticoids in patients
with severe community-acquired pneumonia requiring ICU admission. Three studies with
a combined 800 patients were included. All studies compared corticosteroid therapy
versus placebo over 5-10 days. Outcomes included 30-day mortality, time to clinical
stability, and adverse events including hyperglycemia and secondary infections.
Meta-analysis of mortality outcomes yielded a pooled odds ratio of 0.71 suggesting
a potential benefit of corticosteroids on survival. However rates of hyperglycemia
were significantly elevated in the treatment group.
"""

inputs = tokenizer(
    test_text.strip(),
    max_length=512,
    truncation=True,
    return_tensors='pt'
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=150,
        min_length=40,
        num_beams=4,
        length_penalty=1.5,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("GENERATED SUMMARY:")
print(summary)

GENERATED SUMMARY:
A systematic review of r and omized controlled trials ( RCTs ) evaluating the efficacy of glucocorticoids in the treatment of acute respiratory infections ( AIs ) was performed. A total of 18 studies were included in this systematic review, of which 11 studies met the inclusion criteria. The majority of studies met inclusion criteria, and included patients with AIs with a median AIs score of 0.73 ( 95 % CI 0.79–0.89 ). The pooled RCT scores were calculated using a weighted mean difference ( RMD ) of 0.73 ( CI 0.89–0.89 ), and the pooled RMD scores were 0.91 ( 95% CI 0 to 0.98 )
